In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import time

In [0]:

CAT = "projeto_engenharia"
SCHEMA = "bronze"

def gerar_batch(batch_num):
    df = spark.range(batch_num * 10, (batch_num + 1) * 10).select(
        F.col("id").cast(IntegerType()).alias("InvoiceId"),
        F.expr("CAST((id % 59) + 1 AS INT)").alias("CustomerId"),
        F.expr("CAST((id % 4)  + 1 AS INT)").alias("BillingAddressId"),
        F.current_timestamp().alias("InvoiceDate"),
        F.round(F.rand() * 100, 2).alias("Total"),
        F.lit("streaming").alias("_source"),
        F.current_timestamp().alias("_loaded_at")
    )
    
    df.write.format("delta") \
        .mode("append") \
        .saveAsTable(f"{CAT}.{SCHEMA}.invoice_stream")
    
    print(f"✅ Batch {batch_num + 1} inserido — {10} linhas novas!")

# Simula 3 batches com intervalo de 30 segundos
for i in range(3):
    gerar_batch(i)
    if i < 2:
        print("⏳ Aguardando 30 segundos...")
        time.sleep(30)

print("🎉 Simulação finalizada!")

In [0]:
spark.sql(f"""
    SELECT COUNT(*) as total_linhas, 
           MAX(_loaded_at) as ultima_chegada
    FROM {CAT}.{SCHEMA}.invoice_stream
""").show()